In [ ]:
""" LangExtract Product Information Extraction Example"""


from concurrent.futures import ThreadPoolExecutor
from Modules.hyperparameters import get_hyperparameters
from Modules.load_db_cleaning import join_datasets
from Modules.load_db import get_dataframe_dict
import json
import langextract as lx
import textwrap
import pandas as pd

In [ ]:
""" Constants """

DATASET = 'jsonllm'
HYPER = get_hyperparameters()
# Set to 0 to process all entries
STARTING_POINT = HYPER[DATASET]['starting_point']
# Number of threads for parallel processing
MAX_WORKERS = HYPER[DATASET]['workers']
# Output file path
OUTPUT_PATH = HYPER[DATASET]['output_path']
# Examples data for prompt
EXAMPLES_DATA = HYPER['prompts']['examples_langextract']


PROMPT = textwrap.dedent(HYPER['prompts']['system']['instruction'])

In [ ]:
""" Functions: get_lex_Example, get_examples, check_last_jsonl_id """


def get_lex_Example(example_item: dict) -> lx.data.ExampleData:
    return lx.data.ExampleData(
        text=example_item['text'],
        extractions=[
            lx.data.Extraction(
                extraction_class=example_item['class'],
                extraction_text=example_item['ext_text'],
                attributes=example_item['attributes'],
            ),
        ],
    )


def get_examples() -> list[lx.data.ExampleData]:
    """ Generate example data for LangExtract. """
    examples = []
    for example_item in EXAMPLES_DATA:
        examples.append(get_lex_Example(example_item))
    return examples


def check_last_jsonl_id() -> int:
    """ Check the last processed ID in the output file to resume processing. """
    try:
        with open(OUTPUT_PATH, "r") as f:
            lines = f.readlines()
            if lines:
                starting_point = len(lines) - 1
                if HYPER[DATASET]['verbose']:
                    print(f"Resuming from index: {starting_point}")
                # last_record = json.loads(lines[-2])
                # starting_point = last_record.get("id", -1) + 1
                return starting_point
    except FileNotFoundError:
        return STARTING_POINT
    return STARTING_POINT

In [ ]:
""" Second Round of Constants """

STARTING_POINT = check_last_jsonl_id()

datasets_dict = get_dataframe_dict(['ae-110k', 'oa-mine', 'mave'])
JSONLLM = join_datasets(datasets_dict)
# SAFE_TEXTS = JSONLLM['text'][STARTING_POINT:].tolist()

In [ ]:
""" More Functions """


def save_record_to_file(idx: int, record: dict, file) -> None:
    """ Save a single record to the output file. """
    file.write(json.dumps(record, ensure_ascii=False) + "\n")
    if record and "error" not in record:
        print(f'[{idx}] OK')
    else:
        print(f'[{idx}] ERROR: {record}')


def format_output(idx: int, row: pd.Series, extraction_result: dict = {}, error_msg=None) -> dict:
    """ Format the extraction result into the desired output structure. """

    attrs = {}
    # Safely try to get attributes
    if extraction_result and extraction_result.extractions:
        attrs = extraction_result.extractions[0].attributes or {}

    # json_answer = str(attrs)  # igual ao df (aspas simples)
    json_answer = json.dumps(attrs, ensure_ascii=False)
    attributes = list(attrs.keys())
    # values_text = " | ".join(map(str, values)) # Did anything used this? Maybe ae110k?
    attributes_values = " | ".join(
        f"attribute: {k}, value: {v}" for k, v in attrs.items()
    )
    values = list(attrs.values())

    # candidate_example should e None instead of NaN
    if 'candidate_example' in row and pd.isna(row['candidate_example']):
        row['candidate_example'] = None

    # Same to candidate_text
    if 'candidate_text' in row and pd.isna(row['candidate_text']):
        row['candidate_text'] = None

    record = {
        # original fields
        'unique_id': row.get('unique_id', None),
        'id': row.get('id', None),
        'dataset': row.get('dataset', None),
        'split': row.get('split', None),
        'text': row.get('text', None),
        'source': row.get('source', None),
        'category': row.get('categories', None),
        'similarity_score': row.get('similarity_score', None),
        'candidate_example': row.get('candidate_example', None),
        'candidate_text': row.get('candidate_text', None),
        # not implemented,
        'values_indices': error_msg if error_msg else None,
        # new fields
        'json_answer': str(json_answer),
        'attributes': str(attributes),
        'attributes_values': str(attributes_values),
        'values': str(values),
    }
    return record


def extract_text(idx: int, row: pd.Series) -> tuple[int, dict | None]:
    result = None

    # text = row.get('text', None)
    # Maybe safer somehow
    text = row.get('text', None) if hasattr(
        row, 'get') else getattr(row, 'text', None)

    # Text shouldn't be None in any case, but just
    if text is None or not isinstance(text, (str, int, float)):
        error_msg = "Invalid or missing text input"
        print(f'[{idx}] ERROR: {error_msg}')
        formatted_record = format_output(idx, row, result, error_msg)
        return idx, formatted_record

    try:
        result = lx.extract(
            text_or_documents=text,
            prompt_description=PROMPT,
            examples=get_examples(),
            model_id=HYPER[DATASET]['model_id'],
            model_url=HYPER[DATASET]['model_url'],
            fence_output=False,
            # max_workers=10,
            use_schema_constraints=False,
            language_model_params={"timeout": 900}
        )
        # print(f'[{idx}] Extraction result: {result}')

        # if not result.extractions:
        # return idx, None
        # No need for None lines. This makes post-processing needed later.
        record = format_output(idx, row, result)
        return idx, record
    except Exception as err:
        error_record = format_output(idx, row, result, str(err))
        print(f'[{idx}] EXCEPTION: {err}')
        return idx, error_record

In [ ]:
""" Testing """

TESTING_PATH = 'Generations/jsonllm_test.jsonl'


def testing():
    # iterate rows as (idx, Series) so extract_text can use row.get(...) and row['text']
    rows = JSONLLM.iloc[0:50].copy()
    with open(TESTING_PATH, "a", encoding="utf-8") as file:
        for idx, row in rows.iterrows():
            # print(row.text)
            idx, record = extract_text(idx, row)
            save_record_to_file(idx, record, file)

# line = JSONLLM.iloc[2].copy()
# line['text']
# extract_text(0, JSONLLM.iloc[2].copy())  # test single call
# testing()

In [ ]:
""" Paralelização """


def parallel_extraction(workers: int = MAX_WORKERS, output_path: str = OUTPUT_PATH):
    """ Perform parallel extraction and save results to output file. """
    print(f'Saving results to {output_path} using {workers} workers.')
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor, open(output_path, "a") as f:
        for i, record in executor.map(lambda args: extract_text(*args), JSONLLM[STARTING_POINT:].iterrows()):
            # for i, record in executor.map(lambda args: extract_text(*args), enumerate(SAFE_TEXTS, start=STARTING_POINT)):
            save_record_to_file(i, record, f)

In [ ]:
parallel_extraction()